# SFT model

Перефразирования кликбейт-заголовков для датасета выполнены вручную

In [ ]:
import torch
import pandas as pd
from torch.utils.data import DataLoader, Dataset
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "ai-forever/rugpt3small_based_on_gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained(model_name).to(device)
model.train()

df = pd.read_csv('paraphrased.csv', encoding='utf-8')
sft_data = list(zip(df['Заголовок'], df['Перефразирование']))


class ParaphraseDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        orig, para = self.data[idx]
        full_text = orig + self.tokenizer.eos_token + para + self.tokenizer.eos_token
        enc = self.tokenizer(full_text, truncation=True, max_length=self.max_len, return_tensors='pt')
        input_ids = enc['input_ids'].squeeze()
        attention_mask = enc['attention_mask'].squeeze()
        labels = input_ids.clone()
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels
        }

def collate_batch(batch):
    max_len = max([item['input_ids'].size(0) for item in batch])
    def pad_sequence(seq, max_len, pad_value):
        if seq.size(0) < max_len:
            padded = torch.cat([seq, torch.full((max_len - seq.size(0),), pad_value, dtype=seq.dtype)])
        else:
            padded = seq
        return padded

    input_ids = torch.stack([pad_sequence(item['input_ids'], max_len, tokenizer.pad_token_id) for item in batch])
    attention_mask = torch.stack([pad_sequence(item['attention_mask'], max_len, 0) for item in batch])
    labels = torch.stack([pad_sequence(item['labels'], max_len, -100) for item in batch])
    return {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': labels}

dataset = ParaphraseDataset(sft_data, tokenizer, max_len=128)
loader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate_batch)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
num_epochs = 5

for epoch in range(num_epochs):
    total_loss = 0
    for batch in tqdm(loader, desc=f"SFT Epoch {epoch+1}"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()
    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1} average loss: {avg_loss:.4f}")

model.eval()
model.save_pretrained("rugpt3_sft_paraphrase")
tokenizer.save_pretrained("rugpt3_sft_paraphrase")
print("SFT модель сохранена")

# Classifier reward model

Заголовки перефразировались с помощью sft модели, оценивала выполнялась вручную согласно критериям оценивания сгенерированных в процессе обучения модели заголовков

In [ ]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

df = pd.read_csv('sft_paraphrases_labeled.csv', encoding='utf-8')

class RewardDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        orig = str(row['исходный_заголовок'])
        gen = str(row['перефразирование'])
        label = float(row['оценка'])
        enc = self.tokenizer(
            orig, gen,
            truncation=True,
            max_length=self.max_len,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.float)
        }

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "DeepPavlov/rubert-base-cased-conversational"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1).to(device)

dataset = RewardDataset(df, tokenizer)
loader = DataLoader(dataset, batch_size=16, shuffle=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

model.train()
for epoch in range(5):
    total_loss = 0
    for batch in tqdm(loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels.unsqueeze(1))
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} loss: {total_loss/len(loader):.4f}")

model.save_pretrained("reward_model")
tokenizer.save_pretrained("reward_model")

# RL model

In [ ]:


!pip install bert_score

In [ ]:
import nltk
nltk.download('punkt_tab')

In [ ]:
import pandas as pd
import numpy as np
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import gymnasium as gym
from gym import spaces
from gym.spaces.discrete import Discrete
import torch
from transformers import GenerationConfig
import torch.nn as nn
from transformers import AutoModelForMaskedLM, AutoTokenizer
from IPython.display import clear_output
from tqdm import tqdm
import matplotlib.pyplot as plt
from nltk.tokenize import word_tokenize
import editdistance

# Инициализация модели sft и токенизатора
model_path = "/content/sft_model"
tokenizer = GPT2Tokenizer.from_pretrained(model_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GPT2LMHeadModel.from_pretrained(model_path).to(device)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
class TextGenerationEnv(gym.Env):
    def __init__(self, tokenizer, texts, max_length=60):
        super().__init__()
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.action_space = spaces.Discrete(tokenizer.vocab_size)
        self.observation_space = spaces.Box(low=0, high=tokenizer.vocab_size,
                                            shape=(max_length,), dtype=np.int32)
        self.current_text = None
        self.generated_ids = []
        self.current_step = 0

    def reset(self, idx):
        self.current_text = torch.tensor(self.tokenizer.encode(self.texts[idx]),
                                         dtype=torch.long).to(device)
        self.generated_ids = []
        self.current_step = 0
        return self._get_state()

    def _get_state(self):
        full = self.current_text.cpu().tolist() + self.generated_ids
        if len(full) > self.max_length:
            full = full[:self.max_length]
        else:
            full = full + [self.tokenizer.pad_token_id] * (self.max_length - len(full))
        return np.array(full, dtype=np.int32)

    def step(self, action):
        self.generated_ids.append(action)
        self.current_step += 1
        done = (self.current_step >= (self.max_length - len(self.current_text))) or \
               (action == self.tokenizer.eos_token_id)
        return self._get_state(), 0, done, {}

    def decode_generated(self):
        return self.tokenizer.decode(self.generated_ids, skip_special_tokens=True).strip()

    def decode_input(self):
        return self.tokenizer.decode(self.current_text, skip_special_tokens=True).strip()

In [ ]:
df = pd.read_csv('random_sample_1000_rows.csv', encoding='utf8')
texts = list(df['Заголовок'].values)

env = TextGenerationEnv(tokenizer, texts, max_length=100)

In [ ]:
#генерация заголовков
def generate_headlines(model, input_texts, num_headlines=1, num_beams=4, max_new_tokens=50):
    headlines = []
    for text in input_texts:
        prompt = text + tokenizer.eos_token
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=128).to(device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                num_beams=num_beams,
                repetition_penalty=1.15,
                no_repeat_ngram_size=3,
                early_stopping=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
        decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)
        parts = decoded.split(tokenizer.eos_token)
        if len(parts) >= 2:
            paraphrase = parts[1].strip()
        else:
            paraphrase = decoded.strip()
        paraphrase = paraphrase.replace(tokenizer.eos_token, '').strip()
        headlines.append(paraphrase)
    return headlines

# Функция для оценки модели
def evaluate_model(model, texts, reward_func):
    total_reward = 0
    for text in texts:
        generated_text = generate_headlines(model, [text], num_headlines=1)[0]
        reward = reward_func(text, generated_text)
        total_reward += reward
    return total_reward / len(texts)

In [ ]:
# Функция для отбора лучших моделей
def select_best_models(models, texts, reward_func, num_best=2):
    rewards = [evaluate_model(model, texts, reward_func) for model in models]
    best_models = sorted(zip(rewards, models), key=lambda x: x[0], reverse=True)[:num_best]
    return [m for r, m in best_models]

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

reward_model = AutoModelForSequenceClassification.from_pretrained("/content/reward_model").to(device)
reward_tokenizer = AutoTokenizer.from_pretrained("/content/reward_model")

def classifier_reward(input_texts, generated_texts):
    rewards = []
    for inp, gen in zip(input_texts, generated_texts):
        inputs = reward_tokenizer(inp, gen, return_tensors='pt', truncation=True, max_length=128).to(device)
        with torch.no_grad():
            logits = reward_model(**inputs).logits
            score = torch.sigmoid(logits).squeeze().item()
        rewards.append(score)
    return rewards

In [ ]:
#дополнительные функции для наград
import re
import bert_score
scorer = bert_score.BERTScorer(lang="ru", device=device)

def tokenize_words(text):
    return re.findall(r"[а-яa-zё0-9]+", text.lower())

def normalized_edit_similarity(a, b):
    a = a.lower()
    b = b.lower()
    if not a and not b:
        return 1
    dist = editdistance.eval(a, b)
    return 1 - dist / max(len(a), len(b), 1)

In [ ]:
def compute_reward_1(input_text: str, generated_text: str, human_feedback: float = None):
    #штраф за пустой заголовок
    input_text = input_text.strip()
    generated_text = generated_text.strip()
    if not generated_text or len(generated_text) < 5:
        return -1

    # Оценка BERT Score
    try:
        _, _, f1 = scorer.score([generated_text], [input_text])
        bert_score_value = float(f1[0].item())
    except:
        bert_score_value = 0

    # perplexity
    try:
        with torch.no_grad():
            inputs = tokenizer(generated_text, return_tensors='pt', truncation=True, max_length=128).to(device)
            outputs = model(**inputs)
            logits = outputs.logits
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = inputs["input_ids"][..., 1:].contiguous()
            loss_fct = torch.nn.CrossEntropyLoss(reduction='mean')
            loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
            perplexity = torch.exp(loss).item()
            normalized_perplexity = max(0, 1 - (np.log(perplexity) / 10))
    except:
        normalized_perplexity = 0.5

    # расстояние Левенштейна
    lev_sim = normalized_edit_similarity(input_text, generated_text)

    # Оценка за достаточную длину
    len_score = 0.5 if len(generated_text.split()) >= 5 else -0.5

    # Человеческая обратная связь
    if human_feedback is not None:
        human_score = human_feedback
    else:
        human_score = 0

    final_reward = (
        0.2 * bert_score_value +
        0.2 * normalized_perplexity +
        0.2 * lev_sim +
        0.2 * len_score +
        0.2 * human_score
    )

    final_reward = max(-1, min(1, final_reward))
    return final_reward

In [ ]:
def compute_reward_2(input_text: str, generated_text: str, human_feedback: float = None):
    #штраф за пустой заголовок
    input_text = input_text.strip()
    generated_text = generated_text.strip()
    if not generated_text or len(generated_text) < 5:
        return -1

    # расстояние Левенштейна
    lev_sim = normalized_edit_similarity(input_text, generated_text)

    # Оценка за достаточную длину
    len_score = 0.5 if len(generated_text.split()) >= 5 else 0

    #Уникальность слов в заголовке
    words = generated_text.split()
    if words:
        unique_ratio = len(set(words)) / len(words)
    else:
        unique_ratio = 0

    # ключевые слова
    src_words = set(tokenize_words(input_text))
    gen_words = set(tokenize_words(generated_text))
    if src_words:
        words_coverage = len(src_words & gen_words) / len(src_words)
    else:
        words_coverage = 0

    # Человеческая обратная связь
    if human_feedback is not None:
        human_score = human_feedback
    else:
        human_score = lev_sim

    final_reward = (
        0.25 * lev_sim +
        0.15 * len_score +
        0.2 * unique_ratio +
        0.2 * words_coverage +
        0.2 * human_score
    )

    final_reward = max(-1, min(1, final_reward))
    return final_reward

In [ ]:
def compute_reward_3(input_text: str, generated_text: str, human_feedback: float = None):
  #штраф за пустой заголовок
    input_text = input_text.strip()
    generated_text = generated_text.strip()
    if not generated_text or len(generated_text) < 5:
        return -1

    # оценка BERTScore
    try:
        _, _, f1 = scorer.score([generated_text], [input_text])
        bert_score_value = float(f1[0].item())
    except:
        bert_score_value = 0

    # perplexity
    try:
        with torch.no_grad():
            inputs = tokenizer(generated_text, return_tensors='pt', truncation=True, max_length=128).to(device)
            outputs = model(**inputs)
            logits = outputs.logits
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = inputs["input_ids"][..., 1:].contiguous()
            loss_fct = torch.nn.CrossEntropyLoss(reduction='mean')
            loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
            perplexity = torch.exp(loss).item()
            normalized_perplexity = max(0, 1 - (np.log(perplexity) / 10))
    except:
        normalized_perplexity = 0.5

    # Уникальность слов в заголовке
    words = generated_text.split()
    if words:
        unique_ratio = len(set(words)) / len(words)
    else:
        unique_ratio = 0

    # Оценка за достаточную длину
    len_score = 0.3 if len(generated_text.split()) >= 5 else -0.5

    # Человеческая обратная связь
    if human_feedback is not None:
        human_score = human_feedback
    else:
        human_score = (bert_score_value + normalized_perplexity) / 2

    final_reward = (0.2 * bert_score_value) + (0.2 * normalized_perplexity) + (0.2 *  unique_ratio)+ (0.4 * human_score) + len_score
    final_reward = max(-1, min(1, final_reward))
    return final_reward


In [ ]:
rewards_list = []

# Обновление функции вознаграждения
def update_reward_function(reward_func, rewards_list, window=100, beta=0.1):
    if not rewards_list:
        return reward_func
    recent = rewards_list[-window:]
    baseline = sum(recent) / len(recent)

    def updated_reward_func(input_text: str, generated_text: str, human_feedback: float = None):
        base = reward_func(input_text, generated_text, human_feedback)
        return base - beta * baseline
    return updated_reward_func

In [ ]:
from tqdm.auto import tqdm
from google.colab import drive
import os
import torch
from torch.distributions import Categorical

drive.mount('/content/gdrive')
google_drive_folder = "TRAINED_MODELS_ON_DISK"
google_drive_save_path = f"/content/gdrive/MyDrive/{google_drive_folder}"
local_save_path = "/content/saved_models"

if not os.path.exists(google_drive_save_path):
    os.makedirs(google_drive_save_path)

if not os.path.exists(local_save_path):
    os.makedirs(local_save_path)

# Определение функций вознаграждения
reward_functions = {
    'compute_reward_1': compute_reward_1,
    'compute_reward_2': compute_reward_2,
    'compute_reward_3': compute_reward_3
}

performance_stats = {name: [] for name in reward_functions.keys()}

num_epochs = 8
best_model = model
rewards_list = []
optimizer = torch.optim.Adam(model.parameters(), lr=1e-6)
env = TextGenerationEnv(tokenizer, texts)

for epoch in tqdm(range(num_epochs), desc="Training epochs"):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")

    # Выбор худшей функции вознаграждения
    if epoch >= 0 and all(len(stats) > 0 for stats in performance_stats.values()):
        averages = {name: np.mean(stats[-100:]) for name, stats in performance_stats.items()}
        worst_function_name = min(averages, key=averages.get)
        worst_function = reward_functions[worst_function_name]
        print(f"Выбрана функция вознаграждения: {worst_function_name}")

        if len(rewards_list) > 0:
            updated_function = update_reward_function(worst_function, rewards_list)
            reward_functions[worst_function_name] = updated_function
            worst_function = updated_function
    else:
        worst_function = next(iter(reward_functions.values()))
        worst_function_name = list(reward_functions.keys())[0]
        print(f"Используется функция: {worst_function_name}")

    epoch_rewards = []

    # RL с classifier_reward
    if epoch == 0:
        for episode in tqdm(range(len(texts)), desc=f"Epoch {epoch+1} episodes"):
            state = env.reset(episode)
            log_probs = []
            done = False
            while not done:
                state_t = torch.tensor(state).to(device).unsqueeze(0)
                logits = best_model(state_t).logits[:, -1, :]
                dist = Categorical(logits=logits)
                action = dist.sample()
                log_probs.append(dist.log_prob(action))
                state, _, done, _ = env.step(action.item())
            generated = env.decode_generated()
            original = env.decode_input()
            reward = classifier_reward([original], [generated])[0]
            if log_probs:
                loss = -torch.stack(log_probs).sum() * reward
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(best_model.parameters(), 1)
                optimizer.step()
            epoch_rewards.append(reward)
            performance_stats[worst_function_name].append(reward)

            for name, func in reward_functions.items():
                perf_reward = func(original, generated)
                performance_stats[name].append(perf_reward)

        rewards_list.extend(epoch_rewards)
        epoch_reward = np.mean(epoch_rewards)
        print(f"\nСредняя награда за эпоху: {epoch_reward:.4f}")

        current_epoch_folder = f"model_epoch_{epoch + 1}"
        full_local_path = os.path.join(local_save_path, current_epoch_folder)
        full_google_drive_path = os.path.join(google_drive_save_path, current_epoch_folder)

        os.makedirs(full_local_path, exist_ok=True)
        os.makedirs(full_google_drive_path, exist_ok=True)

        best_model.save_pretrained(full_local_path)
        tokenizer.save_pretrained(full_local_path)
        best_model.save_pretrained(full_google_drive_path)
        tokenizer.save_pretrained(full_google_drive_path)

    # RL с classifier reward и оценка 200 заголовков
    elif epoch == 1:
        for episode in tqdm(range(len(texts)), desc=f"Epoch {epoch+1} episodes (RL)"):
            state = env.reset(episode)
            log_probs = []
            done = False
            while not done:
                state_t = torch.tensor(state).to(device).unsqueeze(0)
                logits = best_model(state_t).logits[:, -1, :]
                dist = Categorical(logits=logits)
                action = dist.sample()
                log_probs.append(dist.log_prob(action))
                state, _, done, _ = env.step(action.item())
            generated = env.decode_generated()
            original = env.decode_input()
            reward = classifier_reward([original], [generated])[0]
            if log_probs:
                loss = -torch.stack(log_probs).sum() * reward
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(best_model.parameters(), 1)
                optimizer.step()
            epoch_rewards.append(reward)
            performance_stats[worst_function_name].append(reward)

            for name, func in reward_functions.items():
              perf_reward = func(original, generated)
              performance_stats[name].append(perf_reward)

        # Ручная оценка 200 заголовков
        print("\nРучная оценка 200 заголовков")
        input_texts = np.random.choice(texts, size=200, replace=False)
        generated_headlines = generate_headlines(best_model, input_texts, num_headlines=1)

        human_feedback = []
        for i, (input_text, headline) in enumerate(zip(input_texts, generated_headlines)):
          print(f"Исходный: {input_text}")

          feedback = float(input(f"Оцените заголовок '{headline}': "))
          human_feedback.append(feedback)

        for input_text, generated_headline, feedback in zip(input_texts, generated_headlines, human_feedback):
            reward = worst_function(input_text, generated_headline, feedback)
            epoch_rewards.append(reward)
            performance_stats[worst_function_name].append(reward)

            for name, func in reward_functions.items():
              perf_reward = func(input_text, generated_headline, feedback)
              performance_stats[name].append(perf_reward)

        rewards_list.extend(epoch_rewards)
        epoch_reward = np.mean(epoch_rewards)
        print(f"\nСредняя награда за эпоху: {epoch_reward:.4f}")

        current_epoch_folder = f"model_epoch_{epoch + 1}"
        full_local_path = os.path.join(local_save_path, current_epoch_folder)
        full_google_drive_path = os.path.join(google_drive_save_path, current_epoch_folder)

        os.makedirs(full_local_path, exist_ok=True)
        os.makedirs(full_google_drive_path, exist_ok=True)

        best_model.save_pretrained(full_local_path)
        tokenizer.save_pretrained(full_local_path)
        best_model.save_pretrained(full_google_drive_path)
        tokenizer.save_pretrained(full_google_drive_path)

    # RL с обновленной функцией вознаграждения
    elif epoch == 2:
        for episode in tqdm(range(len(texts)), desc=f"Epoch {epoch + 1} Processing Episodes"):
            done = False
            state = env.reset(episode)
            log_probs_list = []
            while not done:
                state_tensor = torch.tensor(state).to(device).unsqueeze(0)
                logits = best_model(state_tensor).logits
                action_probs = torch.softmax(logits[:, -1, :], dim=-1)
                dist = Categorical(action_probs)
                action = dist.sample()
                log_prob = dist.log_prob(action)
                log_probs_list.append(log_prob)

                next_state, _, done, _ = env.step(action.item())
                state = next_state

            generated_text = env.decode_generated()
            input_text = env.decode_input()
            reward = worst_function(input_text, generated_text)

            if log_probs_list:
                policy_loss = torch.stack([-lp * reward for lp in log_probs_list]).sum()
                optimizer.zero_grad()
                policy_loss.backward()
                torch.nn.utils.clip_grad_norm_(best_model.parameters(), 1.0)
                optimizer.step()

            epoch_rewards.append(reward)
            performance_stats[worst_function_name].append(reward)

            for name, func in reward_functions.items():
              perf_reward = func(input_text, generated_text)
              performance_stats[name].append(perf_reward)

        rewards_list.extend(epoch_rewards)
        epoch_reward = np.mean(epoch_rewards)
        print(f"\nСредняя награда за эпоху: {epoch_reward:.4f}")

        # обновление худшей функции
        updated_function = update_reward_function(worst_function, performance_stats[worst_function_name])
        reward_functions[worst_function_name] = updated_function
        worst_function = updated_function

        current_epoch_folder = f"model_epoch_{epoch + 1}"
        full_local_path = os.path.join(local_save_path, current_epoch_folder)
        full_google_drive_path = os.path.join(google_drive_save_path, current_epoch_folder)

        os.makedirs(full_local_path, exist_ok=True)
        os.makedirs(full_google_drive_path, exist_ok=True)

        best_model.save_pretrained(full_local_path)
        best_model.save_pretrained(full_google_drive_path)

        tokenizer.save_pretrained(full_local_path)
        tokenizer.save_pretrained(full_google_drive_path)

    # RL с ручной оценкой 100 заголовков
    elif epoch == 3:
        for episode in tqdm(range(len(texts)), desc=f"Epoch {epoch + 1} Processing Episodes"):
            done = False
            state = env.reset(episode)
            log_probs_list = []
            while not done:
                state_tensor = torch.tensor(state).to(device).unsqueeze(0)
                logits = best_model(state_tensor).logits
                action_probs = torch.softmax(logits[:, -1, :], dim=-1)
                dist = Categorical(action_probs)
                action = dist.sample()
                log_prob = dist.log_prob(action)
                log_probs_list.append(log_prob)

                next_state, _, done, _ = env.step(action.item())
                state = next_state

            generated_text = env.decode_generated()
            input_text = env.decode_input()
            reward = worst_function(input_text, generated_text)

            if log_probs_list:
                policy_loss = torch.stack([-lp * reward for lp in log_probs_list]).sum()
                optimizer.zero_grad()
                policy_loss.backward()
                torch.nn.utils.clip_grad_norm_(best_model.parameters(), 1.0)
                optimizer.step()

            epoch_rewards.append(reward)
            performance_stats[worst_function_name].append(reward)

            for name, func in reward_functions.items():
                perf_reward = func(input_text, generated_text)
                performance_stats[name].append(perf_reward)

        # Ручная оценка 100 заголовков
        print("\nРучная оценка 100 заголовков")
        input_texts = np.random.choice(texts, size=100, replace=False)
        generated_headlines = generate_headlines(best_model, input_texts, num_headlines=1)

        human_feedback = []
        for i, (input_text, headline) in enumerate(zip(input_texts, generated_headlines)):
          print(f"Исходный: {input_text}")
          feedback = float(input(f"Оцените заголовок '{headline}': "))
          human_feedback.append(feedback)

        for input_text, generated_headline, feedback in zip(input_texts, generated_headlines, human_feedback):
            reward = worst_function(input_text, generated_headline, feedback)
            epoch_rewards.append(reward)
            performance_stats[worst_function_name].append(reward)

            for name, func in reward_functions.items():
              perf_reward = func(input_text, generated_headline, feedback)
              performance_stats[name].append(perf_reward)

        rewards_list.extend(epoch_rewards)
        epoch_reward = np.mean(epoch_rewards)
        print(f"\nСредняя награда за эпоху: {epoch_reward:.4f}")

        updated_function = update_reward_function(worst_function, performance_stats[worst_function_name])
        reward_functions[worst_function_name] = updated_function
        worst_function = updated_function

        current_epoch_folder = f"model_epoch_{epoch + 1}"
        full_local_path = os.path.join(local_save_path, current_epoch_folder)
        full_google_drive_path = os.path.join(google_drive_save_path, current_epoch_folder)

        os.makedirs(full_local_path, exist_ok=True)
        os.makedirs(full_google_drive_path, exist_ok=True)

        best_model.save_pretrained(full_local_path)
        best_model.save_pretrained(full_google_drive_path)

        tokenizer.save_pretrained(full_local_path)
        tokenizer.save_pretrained(full_google_drive_path)

    # Обычный RL
    elif epoch in [4, 5, 6]:
        for episode in tqdm(range(len(texts)), desc=f"Epoch {epoch + 1} Processing Episodes"):
            done = False
            state = env.reset(episode)
            log_probs_list = []
            while not done:
                state_tensor = torch.tensor(state).to(device).unsqueeze(0)
                logits = best_model(state_tensor).logits
                action_probs = torch.softmax(logits[:, -1, :], dim=-1)
                dist = Categorical(action_probs)
                action = dist.sample()
                log_prob = dist.log_prob(action)
                log_probs_list.append(log_prob)

                next_state, _, done, _ = env.step(action.item())
                state = next_state

            generated_text = env.decode_generated()
            input_text = env.decode_input()
            reward = worst_function(input_text, generated_text)

            if log_probs_list:
                policy_loss = torch.stack([-lp * reward for lp in log_probs_list]).sum()
                optimizer.zero_grad()
                policy_loss.backward()
                torch.nn.utils.clip_grad_norm_(best_model.parameters(), 1.0)
                optimizer.step()

            epoch_rewards.append(reward)
            performance_stats[worst_function_name].append(reward)

            for name, func in reward_functions.items():
              perf_reward = func(input_text, generated_text)
              performance_stats[name].append(perf_reward)

        rewards_list.extend(epoch_rewards)
        epoch_reward = np.mean(epoch_rewards)
        print(f"\nСредняя награда за эпоху: {epoch_reward:.4f}")

        current_epoch_folder = f"model_epoch_{epoch + 1}"
        full_local_path = os.path.join(local_save_path, current_epoch_folder)
        full_google_drive_path = os.path.join(google_drive_save_path, current_epoch_folder)

        os.makedirs(full_local_path, exist_ok=True)
        os.makedirs(full_google_drive_path, exist_ok=True)

        best_model.save_pretrained(full_local_path)
        best_model.save_pretrained(full_google_drive_path)

        tokenizer.save_pretrained(full_local_path)
        tokenizer.save_pretrained(full_google_drive_path)

    # RL с ручной оценкой 50 заголовков
    elif epoch == 7:
        for episode in tqdm(range(len(texts)), desc=f"Epoch {epoch + 1} Processing Episodes"):
            done = False
            state = env.reset(episode)
            log_probs_list = []
            while not done:
                state_tensor = torch.tensor(state).to(device).unsqueeze(0)
                logits = best_model(state_tensor).logits
                action_probs = torch.softmax(logits[:, -1, :], dim=-1)
                dist = Categorical(action_probs)
                action = dist.sample()
                log_prob = dist.log_prob(action)
                log_probs_list.append(log_prob)

                next_state, _, done, _ = env.step(action.item())
                state = next_state

            generated_text = env.decode_generated()
            input_text = env.decode_input()
            reward = worst_function(input_text, generated_text)

            if log_probs_list:
                policy_loss = torch.stack([-lp * reward for lp in log_probs_list]).sum()
                optimizer.zero_grad()
                policy_loss.backward()
                torch.nn.utils.clip_grad_norm_(best_model.parameters(), 1.0)
                optimizer.step()

            epoch_rewards.append(reward)
            performance_stats[worst_function_name].append(reward)

            for name, func in reward_functions.items():
              perf_reward = func(input_text, generated_text)
              performance_stats[name].append(perf_reward)

        # Ручная оценка 50 заголовков
        print("\nРучная оценка 50 заголовков")
        input_texts = np.random.choice(texts, size=50, replace=False)
        generated_headlines = generate_headlines(best_model, input_texts, num_headlines=1)

        human_feedback = []
        for i, (input_text, headline) in enumerate(zip(input_texts, generated_headlines)):
          print(f"Исходный: {input_text}")
          feedback = float(input(f"Оцените заголовок '{headline}': "))
          human_feedback.append(feedback)

        for input_text, generated_headline, feedback in zip(input_texts, generated_headlines, human_feedback):
            reward = worst_function(input_text, generated_headline, feedback)
            epoch_rewards.append(reward)
            performance_stats[worst_function_name].append(reward)

            for name, func in reward_functions.items():
              perf_reward = func(input_text, generated_headline, feedback)
              performance_stats[name].append(perf_reward)

        rewards_list.extend(epoch_rewards)
        epoch_reward = np.mean(epoch_rewards)
        print(f"\nСредняя награда за эпоху: {epoch_reward:.4f}")

        current_epoch_folder = f"model_epoch_{epoch + 1}"
        full_local_path = os.path.join(local_save_path, current_epoch_folder)
        full_google_drive_path = os.path.join(google_drive_save_path, current_epoch_folder)

        os.makedirs(full_local_path, exist_ok=True)
        os.makedirs(full_google_drive_path, exist_ok=True)

        best_model.save_pretrained(full_local_path)
        best_model.save_pretrained(full_google_drive_path)

        tokenizer.save_pretrained(full_local_path)
        tokenizer.save_pretrained(full_google_drive_path)

    with open(os.path.join(local_save_path, f'reward_episode_{epoch + 1}.txt'), 'w') as f:
        f.write(f"Средняя награда за эпоху {epoch + 1}: {epoch_reward}\n")

    test_orig = texts[0]
    test_gen = generate_headlines(best_model, [test_orig])[0]
    print(f"Пример генерации после эпохи {epoch+1}: '{test_gen}'")

    # Отбор двух лучших моделей
    best_models = select_best_models([best_model], texts, reward_func=worst_function, num_best=2)
    print(f"Лучшие модели на эпохе {epoch + 1}: {best_models}")
    best_model = best_models[0]

final_output_dir = os.path.join(local_save_path, "final_trained_model")
final_google_drive_dir = os.path.join(google_drive_save_path, "final_trained_model")

os.makedirs(final_output_dir, exist_ok=True)
os.makedirs(final_google_drive_dir, exist_ok=True)

best_model.save_pretrained(final_output_dir)
best_model.save_pretrained(final_google_drive_dir)

tokenizer.save_pretrained(final_output_dir)
tokenizer.save_pretrained(final_google_drive_dir)

print("\nМодель полностью обучена и сохранена!")
